# 파인튜닝과 앙상블: 로드맵의 마지막 칸

경량 한국어 인코더를 파인튜닝해 §9의 마지막 비교를 끝내고, 그 예측을 TF-IDF 후보와
섞었을 때를 본다. 마지막으로 **모든 모델이 전원 틀린 경계 사례**를 열어 왜 안 움직이는지
확인한다.

학습 코드는 [`scripts/modeling/finetune.py`](../scripts/modeling/finetune.py)이고 §9.4가
요구한 부품을 PyTorch로 직접 구성했다 — `Dataset`/`DataLoader`, 학습·검증 루프,
AdamW + linear warmup, class weight를 넣은 손실, best-checkpoint 선택, seed 고정.
평가 문서는 학습 내내 보지 않고, 멈출 시점은 검증 문서(8/1/1의 그 1)로만 고른다.

비교 기준은 같은 924건·같은 LODO에서 나온 word+char TF-IDF **0.614**(fold 평균) /
**0.638**(통합 OOF), Dummy 0.219다.

재현 명령:

```
$env:RFP_DATASET_VERSION='v4'; python -m scripts.evaluation.finetune_ensemble
$env:RFP_DATASET_VERSION='v4'; python -m scripts.evaluation.boundary_agreement
```

In [ ]:
from pathlib import Path
import json
import sys

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display
from matplotlib import font_manager

ROOT = next((p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / 'scripts').is_dir()), Path.cwd().resolve())
sys.path.insert(0, str(ROOT))
installed = {font.name for font in font_manager.fontManager.ttflist}
plt.rcParams['font.family'] = next((f for f in ['Malgun Gothic', 'NanumGothic', 'Noto Sans CJK KR', 'DejaVu Sans'] if f in installed), 'DejaVu Sans')
plt.rcParams['axes.unicode_minus'] = False

V4 = ROOT / 'reports/current/v4'
runs = [json.loads(l) for l in (V4 / 'finetune_runs.jsonl').read_text(encoding='utf-8').splitlines() if l.strip()]
ens = json.loads((V4 / 'finetune_results.json').read_text(encoding='utf-8'))
cases = pd.read_csv(V4 / 'boundary_agreement_cases.csv', encoding='utf-8-sig')
TFIDF = {'word+char TF-IDF': 0.614, 'char TF-IDF': 0.608, 'Dummy': 0.219}
print(f"파인튜닝 실행 {sum(1 for r in runs if r['config']['fold'] == -1)}회 · 전원 오답 경계 사례 {len(cases)}건")

In [ ]:
rows = []
for r in runs:
    c = r['config']
    if c['fold'] != -1:
        continue
    rows.append({'모델': c['model'].split('/')[-1], '입력': '마스킹' if c['mask'] else '원문', 'seed': c['seed'],
                 'fold 평균': pd.Series([x['test_macro_f1'] for x in r['results']]).mean(),
                 '통합 OOF': ens['singles'][{'roberta-small': 'FT small', 'roberta-large': 'FT large'}.get(
                     c['model'].split('/')[-1], 'FT base +마스킹' if c['mask'] else f"FT base seed{c['seed']}")]['pooled_macro_f1']})
ft = pd.DataFrame(rows)
display(ft.round(3))
base = ft[(ft['모델'] == 'roberta-base') & (ft['입력'] == '원문')]['fold 평균']
print(f"base 원문 seed 3개: 평균 {base.mean():.3f}, 범위 {base.max() - base.min():.3f}")
print(f"word+char TF-IDF와의 차이 {TFIDF['word+char TF-IDF'] - base.mean():+.3f}  ← seed 범위보다 작다")
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
curves = {f"{r['config']['model'].split('/')[-1]}{' 마스킹' if r['config']['mask'] else ''} s{r['config']['seed']}":
          [h['validation_macro_f1'] for h in r['results'][0]['history']] for r in runs if r['config']['fold'] == -1}
pd.DataFrame(curves, index=range(1, 7)).plot(ax=axes[0], marker='o')
axes[0].set_title('fold 0 검증 곡선 — epoch 3 이후 정체'); axes[0].set_xlabel('epoch'); axes[0].set_ylabel('검증 macro F1')
ladder = ft[ft['입력'] == '원문'].groupby('모델')['fold 평균'].mean().reindex(['roberta-small', 'roberta-base', 'roberta-large'])
ladder.plot.bar(ax=axes[1], color='#457b9d', rot=0)
axes[1].axhline(TFIDF['word+char TF-IDF'], color='#e76f51', linestyle='--', label='word+char TF-IDF 0.614')
axes[1].set_title('모델 크기 사다리'); axes[1].legend(); axes[1].set_ylim(0.5, 0.65)
plt.tight_layout()

In [ ]:
o = ens['overlap']
print(f"word+char 오답 {o['left_errors']}  파인튜닝 오답 {o['right_errors']}  둘 다 틀린 것 {o['both_wrong']}")
print(f"한쪽만 틀린 것 {o['one_wrong']}건 → 오라클 정확도 {o['oracle_accuracy']:.3f}")
table = pd.DataFrame({k: {'통합 OOF': v['pooled_macro_f1'], 'fold 평균': v['fold_mean_macro_f1'],
                          '오답': v['errors'], '경계 혼동': v['boundary_errors'], **v['per_label_f1']}
                      for k, v in {**ens['singles'], **ens['ensembles']}.items()}).T
display(table.sort_values('통합 OOF', ascending=False).head(12).round(3))
print(f"중첩 선택(문서 하나를 빼고 아홉으로 조합 선택) {ens['nested']['macro_f1']:.3f} · 열 번의 선택 {ens['nested']['selected']}")

In [ ]:
summary = cases.groupby('정답').agg(건수=('requirement_uid', 'size'), 원문길이_중앙=('원문길이', 'median'),
                                  복합원가=('cost_basis', lambda s: (s == '복합').mean()),
                                  구축높음=('build_difficulty', lambda s: (s == '높음').mean()))
display(summary.round(3))
fig, ax = plt.subplots(figsize=(7, 4))
for label, group in cases.groupby('정답'):
    ax.scatter(group['원문길이'], [label] * len(group), alpha=0.6, s=60)
ax.set_xlabel('원문 길이(문자)'); ax.set_title('전원 오답 28건 — 정답 계약 쪽이 길다')
plt.tight_layout()
display(cases[['정답', '요구사항명', 'cost_basis', '원문길이']].head(8))

## 읽는 법

- **파인튜닝 단독은 TF-IDF와 갈리지 않는다.** base seed 3개의 fold 평균이 0.573 / 0.609 /
  0.597로 범위가 0.036인데, TF-IDF와의 차이는 0.021이다. **판정에 쓸 차이보다 seed 흔들림이
  크다.** seed 하나만 돌렸으면 0.573을 보고 "졌다"고 잘못 적을 뻔했다.
- **크기 사다리에서 적어둔 예측이 틀렸다.** "용량이 크면 노이즈 라벨을 더 외워 나빠진다"고
  했는데 large가 0.617로 가장 높았다. 다만 seed 하나이고 base의 seed 범위 바로 위라
  확정하지 않는다.
- **두 계열은 다른 것을 틀린다.** word+char 오답 294건과 파인튜닝 282건 중 겹치는 것은
  180건뿐이고, 오라클 상한은 0.805다. 기존 soft voting이 오르지 않았던 이유가 여기서
  드러난다 — 세 후보가 전부 희소 TF-IDF라 **같은 것을 틀렸다.**
- **그래서 섞으면 오른다.** `word+char + FT base + FT large`가 통합 OOF 0.647~0.677(FT
  seed에 따라)로 기준선 0.638을 셋 다 넘는다. 문서 하나를 빼고 아홉으로 조합을 고르는
  중첩 선택도 0.677이고 **열 번 모두 같은 조합**을 골라, 선택이 특정 문서에 기대지 않는다.
- **그래도 경계는 그대로다.** 오답은 294 → 255로 줄었는데 견적↔계약 혼동은 98 → 97이다.
  줄어든 39건은 전부 다른 오류다.

## 전원 오답 28건이 말하는 것

경계 라벨 451건 중 아홉 모델이 한 방향으로 전원 틀린 것이 28건이고, 방향별로 성격이
정반대다.

- **정답이 견적반영인 11건은 어려운 항목이 아니다.** 구축 난이도 높음이 18.2%로 나머지
  견적(59.8%)보다 낮은데, 원문에 `협의`가 36.4%(나머지 8.8%), `승인·허가`가 45.5%(20.6%)로
  나온다. 상주 인력·하자보증 SLA·유지관리 같은 **평범한 원가 항목이 계약 어투로 쓰인**
  경우이고, 모델은 어투를 따라간다.
- **정답이 계약인 17건은 정반대다.** cost=복합 88.2%, 구축 높음 94.1%, 원문 892자로 가장
  크고 어려운 항목이고 GPU·인프라·고급인력이 원문을 채운다. 그런데 `협의`는 11.8%로 다른
  계약 건(39.7%)보다 **오히려 적다.** 불확실성이 쉬운 관용구 대신 `폐쇄망`처럼 **문서 밖
  지식이 있어야 함의를 아는 단어**로 나타나는데, 그 단어는 924건 중 10건에만 등장한다.

사람이 이 건들을 쉽게 판정하는 이유도 같다 — 892자를 세지 않고 단어 하나의 함의에
걸리기 때문이다. **빈도로 읽는 표현과 함의로 읽는 사람의 차이**이지, 모델의 성능 부족이
아니다. 계열을 바꾸고 용량을 세 배로 키우고 입력을 마스킹해도 이 28건이 움직이지 않은
것이 그 증거다.